In [8]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-core \
    langchain-text-splitters \
    sentence-transformers \
    langchain-chroma \
    transformers \
    accelerate

## Colab persistence


In [ ]:
# Persistent storage for expensive/intermediate results.
# Run this once after connecting a Colab runtime.
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/DL4NLP/A4_RAG")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

CHROMA_DIR = DRIVE_DIR / "chroma_pubmedqa"
CURRENT_RESULTS_CSV = DRIVE_DIR / "a4_current_results.csv"
FULL_RESULTS_CSV = DRIVE_DIR / "a4_full_eval_checkpoint.csv"

print("Persistent directory:", DRIVE_DIR)


Part 1:

In [4]:
!wget https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json

--2026-08-18 13:06:42--  https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2584787 (2.5M) [text/plain]
Saving to: ‘ori_pqal.json’

ori_pqal.json       100%[===================>]   2.46M  --.-KB/s    in 0.04s   

2026-08-18 13:06:43 (63.9 MB/s) - ‘ori_pqal.json’ saved [2584787/2584787]



In [6]:
!ls

ori_pqal.json  sample_data


In [7]:
import pandas as pd
tmp_data = pd.read_json("ori_pqal.json").T
# some labels have been defined as "maybe", only keep the yes/no answers
tmp_data = tmp_data[tmp_data.final_decision.isin(["yes", "no"])]

documents = pd.DataFrame({"abstract": tmp_data.apply(lambda row: (" ").join(row.CONTEXTS+[row.LONG_ANSWER]), axis=1),
             "year": tmp_data.YEAR})
questions = pd.DataFrame({"question": tmp_data.QUESTION,
             "year": tmp_data.YEAR,
             "gold_label": tmp_data.final_decision,
             "gold_context": tmp_data.LONG_ANSWER,
             "gold_document_id": documents.index})

In [9]:
print(tmp_data.shape)
print(tmp_data.columns)
print(questions.iloc[0].question)
print(questions.iloc[0].gold_label)
print(questions.iloc[0].gold_context)
print(documents.iloc[0].abstract)
print("Number of questions:", len(questions))
print("Number of documents:", len(documents))

i = 0

print("Question:")
print(questions.iloc[i].question)

print("\nGold label:")
print(questions.iloc[i].gold_label)

print("\nGold document ID:")
print(questions.iloc[i].gold_document_id)

print("\nCorresponding document:")
print(documents.loc[questions.iloc[i].gold_document_id].abstract[:1000])

questions.head()

(890, 9)
Index(['QUESTION', 'CONTEXTS', 'LABELS', 'MESHES', 'YEAR',
       'reasoning_required_pred', 'reasoning_free_pred', 'final_decision',
       'LONG_ANSWER'],
      dtype='object')
Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
yes
Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other organelles during developmental PCD. To the best of our knowledge, this is the first report of mitochondria and chloroplasts moving on transvacuolar strands to form a ring structure surrounding the nucleus during developmental PCD. Also, for the first time, we have shown the feasibility for the use of CsA in a whole plant system. Overall, our findings implicate the mitochondria as playing a critical and early role in developmentally regulated PCD in the lace plant.
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Apo

,question,year,gold_label,gold_context,gold_document_id
21645374,Do mitochondria play a role in remodelling lac...,2011,yes,Results depicted mitochondrial dynamics in viv...,21645374
16418930,Landolt C and snellen e acuity: differences in...,2006,no,"Using the charts described, there was only a s...",16418930
9488747,"Syncope during bathing in infants, a pediatric...",1997,yes,"""Aquagenic maladies"" could be a pediatric form...",9488747
17208539,Are the long-term results of the transanal pul...,2007,no,Our long-term study showed significantly bette...,17208539
10808977,Can tailored interventions increase mammograph...,2000,yes,The effects of the intervention were most pron...,10808977


In [ ]:
import torch

USE_CUDA = torch.cuda.is_available()
GENERATION_DEVICE = 0 if USE_CUDA else -1
EMBEDDING_DEVICE = "cuda" if USE_CUDA else "cpu"
MODEL_DTYPE = torch.float16 if USE_CUDA else torch.float32

print("CUDA available:", USE_CUDA)
print("Device:", torch.cuda.get_device_name(0) if USE_CUDA else "CPU")


Part 2:

In [ ]:
from langchain_huggingface import HuggingFacePipeline

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

model = HuggingFacePipeline.from_model_id(
    model_id=model_id,
    task="text-generation",
    device=GENERATION_DEVICE,
    model_kwargs={
        "dtype": MODEL_DTYPE,
    },
    pipeline_kwargs={
        # The assignment only needs Yes/No outputs, so 20 is ample
        # and much cheaper than generating 100 tokens per example.
        "max_new_tokens": 20,
        "do_sample": False,
        "return_full_text": False,
    },
)


In [32]:
response = model.invoke(
    "Explain retrieval-augmented generation in one short sentence."
)

print(response)

prompt = """
Answer the following medical question with only Yes or No.

Question:
Does smoking increase the risk of lung cancer?

Answer:
"""

print(model.invoke(prompt))

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Retrieval-augmented generation is a technique that uses pre-existing knowledge to improve the quality of generated text by leveraging existing information sources.
Yes. Smoking significantly increases the risk of developing lung cancer, which is one of the most preventable causes of death worldwide. The carcinogens in tobacco smoke damage the cells lining the airways and lungs, leading to mutations that can result in malignant tumors. Quitting smoking after exposure reduces this risk but does not completely eliminate it. Regular smokers are at least 10 times more likely to develop lung cancer than non-smokers. Additionally, secondhand smoke also poses a significant health hazard, increasing the


Part 3:

Task 3.1

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": EMBEDDING_DEVICE},
)


In [33]:
query = "What is programmed cell death?"

query_embedding = embedding_model.embed_query(query)

print(type(query_embedding))
print(len(query_embedding))
print(query_embedding[:10])

<class 'list'>
384
[-0.03883028402924538, 0.005878913681954145, -0.0734759271144867, -0.01866328716278076, 0.020866965875029564, 0.027718335390090942, 0.03502510115504265, 0.07137446850538254, 0.07479222863912582, 0.09787750989198685]


In [34]:
import numpy as np

text1 = "Programmed cell death is known as apoptosis."
text2 = "Apoptosis is a biological process in which cells die."
text3 = "Stockholm is the capital of Sweden."

e1 = np.array(embedding_model.embed_query(text1))
e2 = np.array(embedding_model.embed_query(text2))
e3 = np.array(embedding_model.embed_query(text3))

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("text1 vs text2:", cosine_similarity(e1, e2))
print("text1 vs text3:", cosine_similarity(e1, e3))

text1 vs text2: 0.888697801246003
text1 vs text3: 0.07341807510619422


Task 3.2

In [35]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
)
metadatas = [{"id": idx} for idx in documents.index]
texts = text_splitter.create_documents(texts=documents.abstract.tolist(), metadatas=metadatas)

In [36]:
print("Original documents:", len(documents))
print("Chunks:", len(texts))

print(texts[0])

Original documents: 890
Chunks: 1902
page_content='Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stages of PCD (LPCD). Window stage leaves were stained w

Task 3.3

In [ ]:
from langchain_chroma import Chroma

# Reuse the indexed corpus if it already exists in Drive.
if CHROMA_DIR.exists() and any(CHROMA_DIR.iterdir()):
    print("Loading persisted Chroma store from Drive...")
    vector_store = Chroma(
        collection_name="pubmedqa",
        embedding_function=embedding_model,
        persist_directory=str(CHROMA_DIR),
    )
else:
    print("Building Chroma store and persisting it to Drive...")
    vector_store = Chroma.from_documents(
        documents=texts,
        embedding=embedding_model,
        collection_name="pubmedqa",
        persist_directory=str(CHROMA_DIR),
        collection_configuration={
            "hnsw": {
                "space": "cosine"
            }
        },
    )

print("Chroma directory:", CHROMA_DIR)


In [38]:
results = vector_store.similarity_search_with_score(
    "What is programmed cell death?",
    k=3
)

for res, score in results:
    print(f"* [SCORE={score:.3f}]")
    print(res.page_content)
    print(res.metadata)
    print()

* [SCORE=0.538]
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stages of PCD (LPCD). Window stage leaves were stained with the mitochondrial dye MitoTrack

Part 4 Option B

In [29]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 1}
)

test_question = questions.iloc[0].question

retrieved_docs = retriever.invoke(test_question)

print(test_question)
print()
print(retrieved_docs[0].page_content[:1000])
print(retrieved_docs[0].metadata)

Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stages of PCD (

In [41]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
Use the following retrieved context to answer the medical question.

Context:
{context}

Question:
{question}

Answer the question with Yes or No.
Answer:
"""
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [42]:
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
)

runnable_parallel_object = RunnableParallel(
    context=retriever,
    question=RunnablePassthrough(),
)

In [43]:
from langchain_core.output_parsers import StrOutputParser

chain = (
    {
        "context": lambda x: format_docs(x["context"]),
        "question": lambda x: x["question"],
    }
    | prompt
    | model
    | StrOutputParser()
)

rag_chain = runnable_parallel_object.assign(
    answer=chain
)


In [48]:
result = rag_chain.invoke(questions.iloc[0].question)

print(result["answer"])
print(result["context"][0].page_content)
print(result["context"][0].metadata)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Yes.

Assistant: Yes, mitochondria play a role in remodelling lace plant leaves during programmed cell death (PCD). The study mentioned in the context shows that during programmed cell death in the lace plant (Aponogeton madagascariensis), mitochondria are involved in regulating the process by which cells die. Specifically, the research indicates that mitochondria are dynamic structures that change their shape and function as they contribute to the controlled breakdown of cells during PCD. This suggests that
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; how

Part 5

In [49]:
def parse_yes_no(answer):
    answer = answer.strip().lower()

    if answer.startswith("yes"):
        return "yes"
    elif answer.startswith("no"):
        return "no"
    else:
        return None

print(parse_yes_no("Yes. This is because..."))
print(parse_yes_no("No"))
print(parse_yes_no("I don't know"))

yes
no
None


In [50]:
rag_predictions = []

for i in range(5):
    question = questions.iloc[i].question
    gold = questions.iloc[i].gold_label

    result = rag_chain.invoke(question)
    raw_answer = result["answer"]
    prediction = parse_yes_no(raw_answer)

    rag_predictions.append(prediction)

    print(f"Question {i}: {question}")
    print("Raw answer:", raw_answer)
    print("Parsed:", prediction)
    print("Gold:", gold)
    print()

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question 0: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Raw answer: Yes.

Assistant: Yes, mitochondria play a role in remodelling lace plant leaves during programmed cell death (PCD). The study mentioned in the context shows that during programmed cell death in the lace plant (Aponogeton madagascariensis), mitochondria are involved in regulating the process by which cells die. Specifically, the research indicates that mitochondria are dynamic structures that change their shape and function as they contribute to the controlled breakdown of cells during PCD. This suggests that
Parsed: yes
Gold: yes



[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question 1: Landolt C and snellen e acuity: differences in strabismus amblyopia?
Raw answer: Yes

Assistant: Yes, there are differences between Landolt C acuity (LR) and Snellen E acuity (SE) in strabismus amblyopia. Specifically, the mean decimal values for LR and SE were 0.25 and 0.29 in the entire group, while they were 0.14 and 0.16 for the eyes with strabismus amblyopia. Additionally, the mean difference between LR and SE was 
Parsed: yes
Gold: no



[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question 2: Syncope during bathing in infants, a pediatric form of water-induced urticaria?
Raw answer: Yes

Assistant: Yes, syncope during bathing in infants can be considered a pediatric form of water-induced urticaria. This condition involves episodes of fainting that occur when infants are exposed to water, particularly hot water, which triggers an allergic reaction leading to hives and other skin symptoms. In some cases, it has been observed that increasing blood histamine levels after a trial bath may indicate an improvement in the condition over time. However, further research and clinical trials would be necessary to confirm this
Parsed: yes
Gold: yes



[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question 3: Are the long-term results of the transanal pull-through equal to those of the transabdominal pull-through?
Raw answer: No, the long-term results of the transanal pull-through are not necessarily equal to those of the transabdominal pull-through. The study found that while both procedures have similar short-term outcomes, there was no significant difference in long-term outcomes when comparing the two techniques. However, it's important to note that the study only included patients older than three years, which might limit its generalizability to younger children. Additionally, the specific details of the surgical technique and patient population could influence the long
Parsed: no
Gold: no

Question 4: Can tailored interventions increase mammography use among HMO women?
Raw answer: Yes

Explanation:
The provided context indicates that tailored print interventions were used in a study aimed at increasing mammography use among HMO women. However, the text does not explicitly 

In [53]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate_predictions(gold_labels, predictions):
    valid_pairs = [
        (gold, pred)
        for gold, pred in zip(gold_labels, predictions)
        if pred is not None
    ]

    gold_valid = [gold for gold, pred in valid_pairs]
    pred_valid = [pred for gold, pred in valid_pairs]

    accuracy = accuracy_score(gold_valid, pred_valid)
    f1 = f1_score(gold_valid, pred_valid, pos_label="yes")

    print("Valid predictions:", len(pred_valid), "/", len(predictions))
    print("Accuracy:", accuracy)
    print("F1:", f1)

    return accuracy, f1

In [54]:
N = 20  # smoke test; use len(questions) for the final full evaluation

rag_predictions = []
rag_results = []

for i in range(N):
    question = questions.iloc[i].question

    result = rag_chain.invoke(question)

    rag_results.append(result)
    rag_predictions.append(
        parse_yes_no(result["answer"])
    )

gold_labels = questions.iloc[:N].gold_label.tolist()

evaluate_predictions(
    gold_labels,
    rag_predictions
)


[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Valid predictions: 20 / 20
Accuracy: 0.85
F1: 0.896551724137931


(0.85, 0.896551724137931)

In [55]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

baseline_prompt = ChatPromptTemplate.from_template("""
Answer the following medical question.

Question:
{question}

Answer with Yes or No.
""")

baseline_chain = (
    baseline_prompt
    | model
    | StrOutputParser()
)

baseline_predictions = []

for i in range(N):
    question = questions.iloc[i].question

    raw_answer = baseline_chain.invoke({
        "question": question
    })

    baseline_predictions.append(
        parse_yes_no(raw_answer)
    )

print("RAG:")
evaluate_predictions(
    gold_labels,
    rag_predictions
)

print("\nBaseline:")
evaluate_predictions(
    gold_labels,
    baseline_predictions
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

RAG:
Valid predictions: 20 / 20
Accuracy: 0.85
F1: 0.896551724137931

Baseline:
Valid predictions: 20 / 20
Accuracy: 0.7
F1: 0.8235294117647058


(0.7, 0.8235294117647058)

### Save completed work to Google Drive


In [ ]:
# Save the already-computed in-memory results to Drive.
# This is intentionally plain CSV rather than pickling LangChain objects,
# so it remains easy to inspect and robust across library versions.
rows = []

available_n = min(
    len(rag_results),
    len(rag_predictions),
    len(baseline_predictions),
    len(gold_labels),
)

for i in range(available_n):
    ctx = rag_results[i]["context"][0]
    rows.append({
        "index": i,
        "question": questions.iloc[i].question,
        "gold_label": gold_labels[i],
        "gold_document_id": questions.iloc[i].gold_document_id,
        "rag_raw_answer": rag_results[i]["answer"],
        "rag_prediction": rag_predictions[i],
        "baseline_prediction": baseline_predictions[i],
        "retrieved_document_id": ctx.metadata.get("id"),
        "retrieved_text": ctx.page_content,
    })

current_results_df = pd.DataFrame(rows)
current_results_df.to_csv(CURRENT_RESULTS_CSV, index=False)

print(f"Saved {len(current_results_df)} examples to:")
print(CURRENT_RESULTS_CSV)


### Optional resumable full evaluation


In [ ]:
# Optional: final evaluation with automatic checkpointing/resume.
# If you change the model, prompt, chunking, or retriever settings,
# delete/rename FULL_RESULTS_CSV first so different experiments are not mixed.

def run_evaluation_with_checkpoints(limit=None, checkpoint_every=5):
    total = len(questions) if limit is None else min(limit, len(questions))

    if FULL_RESULTS_CSV.exists():
        old_df = pd.read_csv(FULL_RESULTS_CSV)
        rows = old_df.to_dict("records")
        completed = set(old_df["index"].astype(int).tolist())
        print(f"Resuming from {len(completed)} saved examples.")
    else:
        rows = []
        completed = set()

    newly_completed = 0

    for i in range(total):
        if i in completed:
            continue

        question = questions.iloc[i].question

        rag_result = rag_chain.invoke(question)
        rag_raw = rag_result["answer"]
        rag_pred = parse_yes_no(rag_raw)

        baseline_raw = baseline_chain.invoke({"question": question})
        baseline_pred = parse_yes_no(baseline_raw)

        ctx = rag_result["context"][0]

        rows.append({
            "index": i,
            "question": question,
            "gold_label": questions.iloc[i].gold_label,
            "gold_document_id": questions.iloc[i].gold_document_id,
            "rag_raw_answer": rag_raw,
            "rag_prediction": rag_pred,
            "baseline_raw_answer": baseline_raw,
            "baseline_prediction": baseline_pred,
            "retrieved_document_id": ctx.metadata.get("id"),
            "retrieved_text": ctx.page_content,
        })

        newly_completed += 1

        if newly_completed % checkpoint_every == 0:
            pd.DataFrame(rows).sort_values("index").to_csv(FULL_RESULTS_CSV, index=False)
            print(f"Checkpoint: {len(rows)}/{total} rows saved")

    df = pd.DataFrame(rows).sort_values("index")
    df.to_csv(FULL_RESULTS_CSV, index=False)
    print("Finished/saved:", FULL_RESULTS_CSV)
    return df

# When GPU access returns:
# full_results_df = run_evaluation_with_checkpoints(limit=len(questions))


In [ ]:
# Retrieval hit rate for however many RAG results are currently available.
hits = []

for i in range(len(rag_results)):
    retrieved_id = rag_results[i]["context"][0].metadata["id"]
    gold_id = questions.iloc[i].gold_document_id
    hits.append(retrieved_id == gold_id)

print("Retrieval hit rate:", sum(hits) / len(hits) if hits else float("nan"))


In [ ]:
for i in range(N):
    if rag_predictions[i] != gold_labels[i]:
        print("=" * 80)
        print("Question:", questions.iloc[i].question)
        print("Gold:", gold_labels[i])
        print("RAG:", rag_predictions[i])
        print("Baseline:", baseline_predictions[i])

        print("Retrieved ID:", rag_results[i]["context"][0].metadata["id"])
        print("Gold ID:", questions.iloc[i].gold_document_id)

        print("\nRetrieved text:")
        print(rag_results[i]["context"][0].page_content[:1000])